In [3]:
import numpy as np
import pandas as pd

# =====================================================================
# 1. DATA IMPORTING (Directly from Web Source)
# =====================================================================

# Public URL for the official Telco Customer Churn dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

# Load dataset directly into pandas
df = pd.read_csv(url)

print("Dataset successfully loaded!")
print("Initial Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

# =====================================================================
# 2. DATA CLEANING
# =====================================================================

# Drop customerID as it is a unique identifier with no predictive value
if "customerID" in df.columns:
    df.drop("customerID", axis=1, inplace=True)

# TotalCharges contains blank spaces (' ') that cause it to load as an object/string
# Convert whitespace/invalid strings to NaN and force numeric conversion
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].astype(str).str.strip(), errors="coerce"
)

# Impute missing TotalCharges values (due to new customers with 0 tenure) using median
median_total = df["TotalCharges"].median()
df["TotalCharges"].fillna(median_total, inplace=True)

# Convert target variable 'Churn' from Yes/No strings to binary 1/0
if "Churn" in df.columns and df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# =====================================================================
# 3. CATEGORICAL ENCODING & PRE-PROCESSING
# =====================================================================

# Identify categorical and numerical features
cat_cols = [c for c in df.columns if df[c].dtype == "object" and c != "Churn"]
num_cols = [
    c for c in df.columns if df[c].dtype in ["int64", "float64"] and c != "Churn"
]

print("\nCategorical variables to dummy encode:", cat_cols)
print("Numerical variables:", num_cols)

# Apply One-Hot Encoding (Dummy Encoding) to categorical attributes
df_cleaned = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Convert boolean dummy columns to binary integers (1/0)
bool_cols = df_cleaned.select_dtypes(include=["bool"]).columns
df_cleaned[bool_cols] = df_cleaned[bool_cols].astype(int)

# =====================================================================
# 4. FINAL CLEANED DATASET SUMMARY
# =====================================================================

print("\nProcessing complete. Cleaned Dataset Shape:", df_cleaned.shape)
print("\nData Overview:")
print(df_cleaned.info())

Dataset successfully loaded!
Initial Shape: (7043, 21)

First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...              

/var/folders/4_/9_yqxxr929l0p4g22vgxc7km0000gn/T/ipykernel_44831/642212447.py:35: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["TotalCharges"].fillna(median_total, inplace=True)
